In [1]:

import hydra
from hydra import initialize, compose

import torch
import lightning as L

/home/enego/Documents/masters/RayRepresentationTesting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with initialize(config_path="em3rf/configs"):
    cfg=compose(config_name="eval.yaml", overrides=["experiment=denoiser_flow_matching"])
    model: L.LightningModule = hydra.utils.instantiate(cfg.get("model"))

    state_dict = torch.load("E-M3RFfinal.pth", map_location="cpu", weights_only=False)["state_dict"]
    model.load_state_dict(state_dict)

/tmp/ipykernel_3693007/3758938296.py:1: Hydra14MigrationWarning: 
The version_base parameter is not specified.
Hydra will assume defaults for version 1.1.
Hydra 1.4 will remove this compatibility behavior.
Before upgrading to Hydra 1.4, set version_base="1.3" and test your application.
See https://hydra.cc/docs/upgrades/1.3_to_1.4/prepare_for_1_4/ for preparation instructions.
  with initialize(config_path="em3rf/configs"):


❌ No-Overlap Loss is DISABLED.


In [28]:
model.feature_extractor.trainer.save_checkpoint("e-m3rfFracSeg.ckpt")

RuntimeError: FracSeg is not attached to a `Trainer`.

In [36]:
trainer = L.Trainer(accelerator="gpu",max_epochs=0, logger=False, num_sanity_val_steps=0)

TypeError: Trainer.__init__() got an unexpected keyword argument 'datamodule'

In [44]:
trainer.fit(model, datamodule=dataset_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ feature_extractor │ FracSeg             │  1.9 M │ eval  │     0 │
│ 1 │ denoiser          │ DenoiserTransformer │ 43.5 M │ train │     0 │
└───┴───────────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 43.5 M                                                                                           
Non-trainable params: 1.9 M                                                                                        
Total params: 45.4 M                                                                                               
Total estimated model params size (MB): 181.561                                                                    
Modules in train mode: 222                                                                                         
Modules in eval mode: 181                                                                                          
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=0` reached.


In [45]:
trainer.save_checkpoint("e-m3rf.ckpt")

`weights_only` was not set, defaulting to `False`.


In [48]:
type(model)

em3rf.assembly.models.denoiser.denoiser_flow_matching.DenoiserFlowMatching

In [49]:
from em3rf.assembly.models.denoiser.denoiser_flow_matching import DenoiserFlowMatching

In [50]:
DenoiserFlowMatching.load_from_checkpoint("e-m3rf.ckpt")

TypeError: DenoiserFlowMatching.__init__() missing 2 required positional arguments: 'noise_scheduler' and 'val_noise_scheduler'

In [46]:
from em3rf.assembly.models.pretraining.frac_seg import FracSeg

In [47]:
type(model.feature_extractor)

em3rf.assembly.models.pretraining.frac_seg.FracSeg

In [42]:
test_model = FracSeg.load_from_checkpoint("e-m3rfFracSeg.ckpt")

TypeError: FracSeg.__init__() missing 3 required positional arguments: 'pc_feat_dim', 'encoder', and 'optimizer'

In [ ]:
trainer.save_checkpoint()

In [3]:
type(model.encoder)

AttributeError: 'DenoiserFlowMatching' object has no attribute 'encoder'

In [4]:
type(model.feature_extractor)

em3rf.assembly.models.pretraining.frac_seg.FracSeg

In [5]:
type(model.feature_extractor.encoder)

em3rf.assembly.backbones.vn_wrapper.FusionBackbone

In [6]:
from DatasetLoading import RepairDatasetLoader

In [7]:
dataset_loader = RepairDatasetLoader(batch_size=2, dataset_type="RandomRotationPointCloudsDataloader",
                                         representation_folder_name="pointclouds2_5k", num_workers=2)
test_dataloader = dataset_loader.test_dataloader()

In [8]:
batch = next(iter(test_dataloader))
points, normals, colors, rotation = batch

In [9]:
points.shape

torch.Size([2, 2500, 3])

In [10]:
i=0

In [23]:
with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=False):
        # Pass the color into the encoder dictionary
        _, encoder_out_dict = model.feature_extractor.encoder(
            {
                "coord": points[i],
                "offset": torch.tensor([points.shape[1]]),
                "feat": torch.cat([points[i], normals[i]], dim=-1),
                "color": colors[i], # Added color here
                "grid_size": torch.tensor(model.feature_extractor.grid_size).to(points.device),
            }
        )
        model_out = encoder_out_dict["feat"]

In [17]:
point.keys()

dict_keys(['feat', 'coord', 'normal', 'batch'])

In [18]:
point["feat"].shape

torch.Size([2500, 64])

In [21]:
point["batch"].shape


torch.Size([2500])

In [24]:
model_out.shape

torch.Size([2500, 64])

In [25]:
model_out.dtype

torch.float32